# Kaplan-Meier Plots

Generate progression-free survival curves stratified by final-selection model risk groups.

Each model has four curves: internal-test low risk, internal-test high risk, external-test low risk, and external-test high risk.

In [1]:
# ============================================================
# 1. Imports and settings
# ============================================================

import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib import font_manager
from scipy import stats

# ============================================================
# Paths
# ============================================================

patient_list_path = Path('/host/e/D/Data/Habitats/Jishuitan/Patient_lists/image_label_info_set123.xlsx')
model_root = Path('/host/d/projects/Habitats/models/Prognosis')
results_out_dir = Path('/host/d/projects/Habitats/results')
results_out_dir.mkdir(parents=True, exist_ok=True)

# ============================================================
# Survival-time settings
# ============================================================

# How to handle rows where Follow_up_time is earlier than Admission_time.
# Options:
#   'swap'  : swap admission/follow-up dates and keep the case. Default.
#   'drop'  : drop negative-duration cases.
#   'error' : raise an error and stop.
negative_time_action = 'swap'

# The clinical endpoint is treated as a 2-year prognosis endpoint.
x_axis_max_months = 24
x_ticks = [0, 6, 12, 18, 24]

# Landmark 2-year DFS handling requested by the user:
#   label=0 -> DFS-free at 24 months, regardless of whether actual follow-up is 21-24 months
#   label=1 with event time <=24 months -> observed event at that time
#   label=1 with event time >24 months -> censored/DFS-free at 24 months
landmark_months = 24

# ============================================================
# Font settings: keep consistent with AUC_reports
# ============================================================

times_font_paths = [
    '/host/c/Windows/Fonts/times.ttf',
    '/host/c/Windows/Fonts/timesbd.ttf',
    '/host/c/Windows/Fonts/timesi.ttf',
    '/host/c/Windows/Fonts/timesbi.ttf',
]
for font_path in times_font_paths:
    if os.path.isfile(font_path):
        font_manager.fontManager.addfont(font_path)

plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42
plt.rcParams['font.family'] = 'Times New Roman'
plt.rcParams['font.serif'] = ['Times New Roman']
plt.rcParams['axes.unicode_minus'] = False

# ============================================================
# Model sources and output files
# ============================================================

model_sources = [
    {
        'name': 'Clinical',
        'kind': 'standard_final_selection',
        'folder': model_root / 'clinical' / 'final_selections',
        'prob_col': 'prob_final_selection',
        'internal_metrics': model_root / 'clinical' / 'final_selections' / 'internal_test_final_selection_metrics.xlsx',
        'external_metrics': model_root / 'clinical' / 'final_selections' / 'external_test_final_selection_metrics.xlsx',
        'single_pdf': results_out_dir / 'KM_Clinical.pdf',
    },
    {
        'name': 'C-radiomics',
        'kind': 'standard_final_selection',
        'folder': model_root / 'whole_image' / 'final_selections',
        'prob_col': 'prob_final_selection',
        'internal_metrics': model_root / 'whole_image' / 'final_selections' / 'internal_test_final_selection_metrics.xlsx',
        'external_metrics': model_root / 'whole_image' / 'final_selections' / 'external_test_final_selection_metrics.xlsx',
        'single_pdf': results_out_dir / 'KM_C_radiomics.pdf',
    },
    {
        'name': 'H-radiomics',
        'kind': 'standard_final_selection',
        'folder': model_root / 'habitats_avg' / 'final_selections',
        'prob_col': 'prob_final_selection',
        'internal_metrics': model_root / 'habitats_avg' / 'final_selections' / 'internal_test_final_selection_metrics.xlsx',
        'external_metrics': model_root / 'habitats_avg' / 'final_selections' / 'external_test_final_selection_metrics.xlsx',
        'single_pdf': results_out_dir / 'KM_H_radiomics.pdf',
    },
    {
        'name': 'DL_3D',
        'kind': 'standard_final_selection',
        'folder': model_root / 'dl_3d_ml_all' / 'final_selections',
        'prob_col': 'prob_final_selection',
        'internal_metrics': model_root / 'dl_3d_ml_all' / 'final_selections' / 'internal_test_final_selection_metrics.xlsx',
        'external_metrics': model_root / 'dl_3d_ml_all' / 'final_selections' / 'external_test_final_selection_metrics.xlsx',
        'single_pdf': results_out_dir / 'KM_DL_3D.pdf',
    },
    {
        'name': 'fusion_soft_vote',
        'kind': 'soft_vote',
        'prediction_path': model_root / 'fusion' / 'soft_vote_predictions.xlsx',
        'prob_col': 'prob_soft_vote',
        'metrics_path': model_root / 'fusion' / 'soft_vote_metrics.xlsx',
        'single_pdf': results_out_dir / 'KM_Soft_vote.pdf',
    },
    {
        'name': 'fusion_stacking',
        'kind': 'standard_final_selection',
        'folder': model_root / 'fusion' / 'stacking' / 'final_selections' / 'RF',
        'prob_col': 'prob_final_selection',
        'internal_metrics': model_root / 'fusion' / 'stacking' / 'final_selections' / 'RF' / 'internal_test_final_selection_metrics.xlsx',
        'external_metrics': model_root / 'fusion' / 'stacking' / 'final_selections' / 'RF' / 'external_test_final_selection_metrics.xlsx',
        'single_pdf': results_out_dir / 'KM_stacking.pdf',
    },
]

combined_pdf_path = results_out_dir / 'KM_combined.pdf'
hazard_ratio_table_path = results_out_dir / 'hazard_ratio_table.xlsx'

print('Patient list:', patient_list_path)
print('Output directory:', results_out_dir)
print('negative_time_action:', negative_time_action)

def save_pdf_and_ppt_safe_svg(fig, pdf_path, **kwargs):
    """Save the normal PDF plus a PPT-friendly SVG copy.

    The SVG suffix is `_ppt_safe.svg`, so original PDF manuscript outputs stay unchanged.
    """
    pdf_path = Path(pdf_path)
    fig.savefig(pdf_path, **kwargs)
    svg_path = pdf_path.with_name(pdf_path.stem + '_ppt_safe.svg')
    fig.savefig(svg_path, format='svg', **kwargs)
    return svg_path


Patient list: /host/e/D/Data/Habitats/Jishuitan/Patient_lists/image_label_info_set123.xlsx
Output directory: /host/d/projects/Habitats/results
negative_time_action: swap


In [2]:
# ============================================================
# 2. Build survival table
# ============================================================

def parse_year_month(value):
    """Parse values such as 2026-05, 2026.5, 2025.11 into month-level Timestamp."""
    if pd.isna(value):
        return pd.NaT
    text = str(value).strip()
    if text == '' or text.lower() == 'nan':
        return pd.NaT

    if '-' in text:
        parts = text.split('-')
        year = int(float(parts[0]))
        month = int(float(parts[1]))
    elif '.' in text:
        year_text, month_text = text.split('.', 1)
        year = int(float(year_text))
        month = int(float(month_text))
    else:
        year = int(float(text))
        month = 1

    if month < 1 or month > 12:
        raise ValueError(f'Invalid month parsed from {value!r}: {month}')
    return pd.Timestamp(year=year, month=month, day=1)


def month_difference(later, earlier):
    return (later.year - earlier.year) * 12 + (later.month - earlier.month)


def build_survival_table():
    df = pd.read_excel(patient_list_path)
    required = ['Patient_set', 'Patient_index', 'Prognosis_label', 'Follow_up_time', 'Admission_time']
    missing = [col for col in required if col not in df.columns]
    if missing:
        raise KeyError(f'Missing required columns from patient list: {missing}')

    keep_cols = required + [col for col in ['Name', 'Sort_order', 'Medical_record_number'] if col in df.columns]
    surv = df[keep_cols].copy()
    surv['event'] = surv['Prognosis_label'].astype(int)
    surv['follow_dt'] = surv['Follow_up_time'].apply(parse_year_month)
    surv['admission_dt'] = surv['Admission_time'].apply(parse_year_month)
    surv['duration_months'] = [
        month_difference(follow, admission) if pd.notna(follow) and pd.notna(admission) else np.nan
        for follow, admission in zip(surv['follow_dt'], surv['admission_dt'])
    ]

    negative_rows = surv[surv['duration_months'] < 0].copy()
    if negative_rows.shape[0] > 0:
        print('Negative duration rows before handling:')
        display_cols = [col for col in ['Patient_set', 'Patient_index', 'Name', 'Prognosis_label', 'Admission_time', 'Follow_up_time', 'duration_months'] if col in negative_rows.columns]
        display(negative_rows[display_cols])

        if negative_time_action == 'swap':
            neg_idx = surv['duration_months'] < 0
            old_admission = surv.loc[neg_idx, 'admission_dt'].copy()
            old_follow = surv.loc[neg_idx, 'follow_dt'].copy()
            surv.loc[neg_idx, 'admission_dt'] = old_follow.values
            surv.loc[neg_idx, 'follow_dt'] = old_admission.values
            surv.loc[neg_idx, 'duration_months'] = [
                month_difference(follow, admission)
                for follow, admission in zip(surv.loc[neg_idx, 'follow_dt'], surv.loc[neg_idx, 'admission_dt'])
            ]
            print('Negative duration rows were handled by swapping admission/follow-up dates.')
            display(surv.loc[neg_idx, display_cols])
        elif negative_time_action == 'drop':
            surv = surv[surv['duration_months'] >= 0].copy()
            print('Negative duration rows were dropped.')
        elif negative_time_action == 'error':
            raise RuntimeError('Negative duration rows found. Set negative_time_action to swap or drop to continue.')
        else:
            raise ValueError(f'Unknown negative_time_action: {negative_time_action}')

    surv = surv.dropna(subset=['duration_months', 'event']).copy()
    surv['raw_duration_months'] = surv['duration_months'].astype(float)
    surv['raw_event'] = surv['event'].astype(int)

    # Convert raw follow-up/event dates to the user-defined 2-year landmark endpoint.
    # For label=0 cases, even if available follow-up is only 21-24 months, the case is
    # treated as DFS-free at 24 months. For label=1 cases, only events within 24 months
    # are counted as observed events; later events are censored at 24 months.
    surv['duration_months'] = landmark_months
    surv['event'] = 0

    event_within_landmark = (
        (surv['raw_event'] == 1)
        & (surv['raw_duration_months'] <= landmark_months)
    )
    surv.loc[event_within_landmark, 'duration_months'] = surv.loc[event_within_landmark, 'raw_duration_months']
    surv.loc[event_within_landmark, 'event'] = 1

    surv['duration_months'] = surv['duration_months'].astype(float)
    surv['event'] = surv['event'].astype(int)

    return surv

survival_df = build_survival_table()
print('Survival table shape:', survival_df.shape)
print('Events:', int(survival_df['event'].sum()), 'Censored:', int((1 - survival_df['event']).sum()))
print('Duration months summary:')
display(survival_df['duration_months'].describe())


Survival table shape: (348, 14)
Events: 90 Censored: 258
Duration months summary:


count    348.000000
mean      21.250000
std        5.476502
min        1.000000
25%       22.750000
50%       24.000000
75%       24.000000
max       24.000000
Name: duration_months, dtype: float64

In [3]:
# ============================================================
# 3. Read model predictions and thresholds
# ============================================================

def get_final_threshold_from_metrics(metrics_path, dataset=None):
    metrics_path = Path(metrics_path)
    if not metrics_path.is_file():
        raise FileNotFoundError(f'Missing metrics file: {metrics_path}')
    df = pd.read_excel(metrics_path)

    if dataset is not None and 'dataset' in df.columns:
        df = df[df['dataset'].astype(str) == dataset].copy()

    if 'is_final_selection' in df.columns:
        final_df = df[df['is_final_selection'].astype(bool)].copy()
        if final_df.shape[0] > 0:
            df = final_df

    if df.shape[0] == 0:
        raise RuntimeError(f'No metric row found in {metrics_path}')
    if 'threshold' not in df.columns:
        raise KeyError(f'No threshold column in {metrics_path}')

    return float(df.iloc[-1]['threshold'])


def load_prediction_for_model(source, cohort):
    if cohort not in ['internal_test', 'external_test']:
        raise ValueError('cohort must be internal_test or external_test')

    if source['kind'] == 'standard_final_selection':
        filename = f'{cohort}_final_selection_predictions.xlsx'
        path = Path(source['folder']) / filename
        threshold_path = source['internal_metrics'] if cohort == 'internal_test' else source['external_metrics']
        threshold = get_final_threshold_from_metrics(threshold_path)
        df = pd.read_excel(path)
    elif source['kind'] == 'soft_vote':
        path = Path(source['prediction_path'])
        dataset = 'internal_test' if cohort == 'internal_test' else 'external_test'
        threshold = get_final_threshold_from_metrics(source['metrics_path'], dataset=dataset)
        df_all = pd.read_excel(path)
        df = df_all[df_all['dataset'].astype(str) == dataset].copy()
    else:
        raise ValueError(f"Unknown source kind: {source['kind']}")

    prob_col = source['prob_col']
    required = ['Patient_set', 'Patient_index', prob_col]
    missing = [col for col in required if col not in df.columns]
    if missing:
        raise KeyError(f'Missing columns {missing} in {path}')

    out = df[required].copy()
    out = out.rename(columns={prob_col: 'probability'})
    out['probability'] = out['probability'].astype(float)
    out['risk_group'] = np.where(out['probability'] >= threshold, 'High risk', 'Low risk')
    out['cohort'] = cohort
    out['threshold'] = threshold
    out['model'] = source['name']
    return out, path


def build_model_km_table(source):
    pieces = []
    trace = []
    for cohort in ['internal_test', 'external_test']:
        pred_df, path = load_prediction_for_model(source, cohort)
        merged = pred_df.merge(
            survival_df,
            on=['Patient_set', 'Patient_index'],
            how='inner',
            validate='one_to_one',
        )
        if merged.shape[0] != pred_df.shape[0]:
            print(f'Warning: model={source["name"]}, cohort={cohort}, prediction rows={pred_df.shape[0]}, merged rows={merged.shape[0]}')
        pieces.append(merged)
        trace.append({
            'model': source['name'],
            'cohort': cohort,
            'prediction_path': str(path),
            'threshold': float(pred_df['threshold'].iloc[0]),
            'n_prediction': pred_df.shape[0],
            'n_merged': merged.shape[0],
            'high_risk_n': int((merged['risk_group'] == 'High risk').sum()),
            'low_risk_n': int((merged['risk_group'] == 'Low risk').sum()),
        })
    return pd.concat(pieces, ignore_index=True), pd.DataFrame(trace)

model_km_tables = {}
trace_tables = []
for source in model_sources:
    table, trace = build_model_km_table(source)
    model_km_tables[source['name']] = table
    trace_tables.append(trace)

km_trace_df = pd.concat(trace_tables, ignore_index=True)
display(km_trace_df)


,model,cohort,prediction_path,threshold,n_prediction,n_merged,high_risk_n,low_risk_n
0,Clinical,internal_test,/host/d/projects/Habitats/models/Prognosis/cli...,0.390000,96,96,20,76
1,Clinical,external_test,/host/d/projects/Habitats/models/Prognosis/cli...,0.413615,64,64,39,25
2,C-radiomics,internal_test,/host/d/projects/Habitats/models/Prognosis/who...,0.283935,96,96,49,47
3,C-radiomics,external_test,/host/d/projects/Habitats/models/Prognosis/who...,0.292894,64,64,26,38
4,H-radiomics,internal_test,/host/d/projects/Habitats/models/Prognosis/hab...,0.278320,96,96,33,63
5,H-radiomics,external_test,/host/d/projects/Habitats/models/Prognosis/hab...,0.290715,64,64,27,37
6,DL_3D,internal_test,/host/d/projects/Habitats/models/Prognosis/dl_...,0.425834,96,96,38,58
7,DL_3D,external_test,/host/d/projects/Habitats/models/Prognosis/dl_...,0.512995,64,64,22,42
8,fusion_soft_vote,internal_test,/host/d/projects/Habitats/models/Prognosis/fus...,0.305003,96,96,46,50
9,fusion_soft_vote,external_test,/host/d/projects/Habitats/models/Prognosis/fus...,0.330827,64,64,34,30


In [4]:
# ============================================================
# 4. Kaplan-Meier and log-rank functions
# ============================================================

def km_step_curve(durations, events, x_max=None):
    durations = np.asarray(durations, dtype=float)
    events = np.asarray(events, dtype=int)

    if len(durations) == 0:
        return np.array([0.0]), np.array([1.0])

    event_times = np.sort(np.unique(durations[events == 1]))
    x = [0.0]
    y = [1.0]
    survival = 1.0

    for t in event_times:
        at_risk = np.sum(durations >= t)
        n_events = np.sum((durations == t) & (events == 1))
        if at_risk <= 0:
            continue
        x.extend([t, t])
        y.extend([survival, survival * (1.0 - n_events / at_risk)])
        survival = y[-1]

    if x_max is None:
        x_max = max(float(np.max(durations)), x_axis_max_months)
    x.append(float(x_max))
    y.append(survival)
    return np.asarray(x), np.asarray(y)


def logrank_p_value(durations_a, events_a, durations_b, events_b):
    durations_a = np.asarray(durations_a, dtype=float)
    durations_b = np.asarray(durations_b, dtype=float)
    events_a = np.asarray(events_a, dtype=int)
    events_b = np.asarray(events_b, dtype=int)

    event_times = np.sort(np.unique(np.concatenate([durations_a[events_a == 1], durations_b[events_b == 1]])))
    observed_a = 0.0
    expected_a = 0.0
    variance_a = 0.0

    for t in event_times:
        n_a = np.sum(durations_a >= t)
        n_b = np.sum(durations_b >= t)
        d_a = np.sum((durations_a == t) & (events_a == 1))
        d_b = np.sum((durations_b == t) & (events_b == 1))
        n_total = n_a + n_b
        d_total = d_a + d_b
        if n_total <= 1 or d_total == 0:
            continue

        expected = d_total * (n_a / n_total)
        # Hypergeometric variance with finite population correction.
        variance = (n_a * n_b * d_total * (n_total - d_total)) / (n_total**2 * (n_total - 1))
        observed_a += d_a
        expected_a += expected
        variance_a += variance

    if variance_a <= 0:
        return np.nan
    chi2 = (observed_a - expected_a) ** 2 / variance_a
    return float(1.0 - stats.chi2.cdf(chi2, df=1))


def format_p_value(p):
    if p is None or pd.isna(p):
        return 'NA'
    if p < 0.001:
        return '<0.001'
    return f'{p:.3f}'


def cox_hr_binary(durations, events, high_risk):
    """Single-variable Cox PH for high-risk vs low-risk.

    This is a small Breslow-tie implementation for a binary covariate.
    It returns HR, 95% CI, and Wald p-value for HR != 1.
    """
    durations = np.asarray(durations, dtype=float)
    events = np.asarray(events, dtype=int)
    x = np.asarray(high_risk, dtype=float)

    mask = np.isfinite(durations) & np.isfinite(events) & np.isfinite(x)
    durations = durations[mask]
    events = events[mask]
    x = x[mask]

    result = {
        'hr': np.nan,
        'hr_ci_low': np.nan,
        'hr_ci_high': np.nan,
        'cox_p': np.nan,
        'coef': np.nan,
        'se': np.nan,
        'status': 'ok',
    }

    if len(durations) == 0:
        result['status'] = 'empty input'
        return result
    if len(np.unique(x)) < 2:
        result['status'] = 'only one risk group'
        return result
    if np.sum(events == 1) == 0:
        result['status'] = 'no events'
        return result

    beta = 0.0
    event_times = np.sort(np.unique(durations[events == 1]))

    info = np.nan
    for _ in range(100):
        score = 0.0
        info = 0.0
        for t in event_times:
            event_at_t = (durations == t) & (events == 1)
            d_t = np.sum(event_at_t)
            if d_t == 0:
                continue
            risk = durations >= t
            x_risk = x[risk]
            eta = np.clip(beta * x_risk, -50, 50)
            w = np.exp(eta)
            s0 = np.sum(w)
            s1 = np.sum(w * x_risk)
            s2 = np.sum(w * x_risk * x_risk)
            if s0 <= 0:
                continue
            mean_x = s1 / s0
            var_x = s2 / s0 - mean_x * mean_x
            score += np.sum(x[event_at_t]) - d_t * mean_x
            info += d_t * var_x

        if info <= 1e-12 or not np.isfinite(info):
            result['status'] = 'unstable information'
            return result

        step = score / info
        step = float(np.clip(step, -2.0, 2.0))
        beta += step
        if abs(step) < 1e-8:
            break

    se = float(np.sqrt(1.0 / info))
    z = beta / se
    p_value = float(2.0 * (1.0 - stats.norm.cdf(abs(z))))

    result.update({
        'hr': float(np.exp(beta)),
        'hr_ci_low': float(np.exp(beta - 1.96 * se)),
        'hr_ci_high': float(np.exp(beta + 1.96 * se)),
        'cox_p': p_value,
        'coef': float(beta),
        'se': se,
        'status': 'ok',
    })
    return result


def format_hr_text(hr_info):
    if hr_info is None or hr_info.get('status') != 'ok':
        return 'NA'
    star = r'$^{*}$' if hr_info['cox_p'] < 0.05 else ''
    return f"{hr_info['hr']:.2f} ({hr_info['hr_ci_low']:.2f}, {hr_info['hr_ci_high']:.2f}){star}"

print('KM, log-rank, and Cox HR functions ready.')


KM, log-rank, and Cox HR functions ready.


In [5]:
# ============================================================
# 5. Plot functions
# ============================================================

curve_styles = {
    ('internal_test', 'Low risk'):  {'color': '#1f77b4', 'linestyle': '-',  'label': 'Internal test Low risk'},
    ('internal_test', 'High risk'): {'color': '#d62728', 'linestyle': '-',  'label': 'Internal test High risk'},
    ('external_test', 'Low risk'):  {'color': '#17becf', 'linestyle': '--', 'label': 'External test Low risk'},
    ('external_test', 'High risk'): {'color': '#ff7f0e', 'linestyle': '--', 'label': 'External test High risk'},
}


def calculate_model_survival_stats(model_name):
    table = model_km_tables[model_name]
    stats_out = {}
    for cohort in ['internal_test', 'external_test']:
        cohort_df = table[table['cohort'] == cohort].copy()
        low = cohort_df[cohort_df['risk_group'] == 'Low risk']
        high = cohort_df[cohort_df['risk_group'] == 'High risk']

        logrank_p = logrank_p_value(
            low['duration_months'].to_numpy(), low['event'].to_numpy(),
            high['duration_months'].to_numpy(), high['event'].to_numpy(),
        )

        high_risk_binary = (cohort_df['risk_group'] == 'High risk').astype(int).to_numpy()
        hr_info = cox_hr_binary(
            cohort_df['duration_months'].to_numpy(),
            cohort_df['event'].to_numpy(),
            high_risk_binary,
        )

        stats_out[cohort] = {
            'logrank_p': logrank_p,
            'hr_info': hr_info,
            'low_n': int(low.shape[0]),
            'high_n': int(high.shape[0]),
            'low_events': int(low['event'].sum()),
            'high_events': int(high['event'].sum()),
            'threshold': float(cohort_df['threshold'].iloc[0]) if cohort_df.shape[0] > 0 else np.nan,
        }
    return stats_out


def plot_km_for_model(model_name, ax=None, show_legend=True, show_xlabel=True, show_ylabel=True):
    table = model_km_tables[model_name]
    if ax is None:
        fig, ax = plt.subplots(figsize=(5.2, 4.7))
    else:
        fig = ax.figure

    survival_stats = calculate_model_survival_stats(model_name)
    for cohort in ['internal_test', 'external_test']:
        for risk_group in ['Low risk', 'High risk']:
            subset = table[(table['cohort'] == cohort) & (table['risk_group'] == risk_group)]
            style = curve_styles[(cohort, risk_group)]
            x, y = km_step_curve(subset['duration_months'].to_numpy(), subset['event'].to_numpy(), x_max=x_axis_max_months)
            ax.step(x, y, where='post', color=style['color'], linestyle=style['linestyle'], linewidth=2.0, label=style['label'])

    ax.set_xlim(0, x_axis_max_months)
    ax.set_ylim(0, 1.02)
    ax.set_xticks(x_ticks)
    ax.set_yticks(np.linspace(0, 1.0, 6))
    if show_xlabel:
        ax.set_xlabel('Time (months)', fontsize=14)
    if show_ylabel:
        ax.set_ylabel('Disease-free probability', fontsize=14)
    ax.set_title(model_name, fontsize=17, fontweight='bold')
    ax.tick_params(axis='both', labelsize=12, direction='in')
    ax.grid(False)
    for spine in ax.spines.values():
        spine.set_linewidth(1.0)

    text = (
        f'internal:\n'
        f'log-rank p = {format_p_value(survival_stats["internal_test"]["logrank_p"])}\n'
        f'HR = {format_hr_text(survival_stats["internal_test"]["hr_info"])}\n'
        f'external:\n'
        f'log-rank p = {format_p_value(survival_stats["external_test"]["logrank_p"])}\n'
        f'HR = {format_hr_text(survival_stats["external_test"]["hr_info"])}'
    )
    ax.text(0.05, 0.30, text, transform=ax.transAxes, fontsize=12.0, fontweight='bold', va='top')

    if show_legend:
        ax.legend(
            loc='lower right',
            bbox_to_anchor=(0.92, 0.05),
            fontsize=11.0,
            frameon=True,
            edgecolor='0.4',
            borderaxespad=0.2,
        )

    return fig, ax, survival_stats

print('Plotting functions ready.')


Plotting functions ready.


In [6]:
# ============================================================
# 6. Save six single-model PDFs and one combined 2 x 3 PDF
# ============================================================

hazard_ratio_rows = []

# Single-model figures.
for source in model_sources:
    model_name = source['name']
    fig, ax, survival_stats = plot_km_for_model(model_name, show_legend=True)
    fig.tight_layout()
    save_pdf_and_ppt_safe_svg(fig, source['single_pdf'], bbox_inches='tight')
    plt.close(fig)
    print('Saved:', source['single_pdf'])

    for cohort in ['internal_test', 'external_test']:
        stats_current = survival_stats[cohort]
        hr_info = stats_current['hr_info']
        hazard_ratio_rows.append({
            'Model': model_name,
            'Cohort': cohort,
            'Threshold': stats_current['threshold'],
            'Low_risk_n': stats_current['low_n'],
            'High_risk_n': stats_current['high_n'],
            'Low_risk_events': stats_current['low_events'],
            'High_risk_events': stats_current['high_events'],
            'Logrank_p': stats_current['logrank_p'],
            'HR': hr_info.get('hr', np.nan),
            'HR_95CI_low': hr_info.get('hr_ci_low', np.nan),
            'HR_95CI_high': hr_info.get('hr_ci_high', np.nan),
            'Cox_p': hr_info.get('cox_p', np.nan),
            'Cox_status': hr_info.get('status', ''),
            'HR_report': format_hr_text(hr_info),
        })

# Combined figure.
fig, axes = plt.subplots(2, 3, figsize=(15.6, 9.2))
axes = axes.ravel()
for i, source in enumerate(model_sources):
    model_name = source['name']
    _, _, _ = plot_km_for_model(
        model_name,
        ax=axes[i],
        show_legend=True,
        show_xlabel=(i >= 3),
        show_ylabel=(i % 3 == 0),
    )

fig.tight_layout(w_pad=1.5, h_pad=2.0)
save_pdf_and_ppt_safe_svg(fig, combined_pdf_path, bbox_inches='tight')
plt.close(fig)
print('Saved combined KM figure:', combined_pdf_path)

hazard_ratio_df = pd.DataFrame(hazard_ratio_rows)
hazard_ratio_df.to_excel(hazard_ratio_table_path, index=False)
print('Saved hazard ratio table:', hazard_ratio_table_path)
display(hazard_ratio_df)


Saved: /host/d/projects/Habitats/results/KM_Clinical.pdf


Saved: /host/d/projects/Habitats/results/KM_C_radiomics.pdf


Saved: /host/d/projects/Habitats/results/KM_H_radiomics.pdf


Saved: /host/d/projects/Habitats/results/KM_DL_3D.pdf


Saved: /host/d/projects/Habitats/results/KM_Soft_vote.pdf


Saved: /host/d/projects/Habitats/results/KM_stacking.pdf


Saved combined KM figure: /host/d/projects/Habitats/results/KM_combined.pdf
Saved hazard ratio table: /host/d/projects/Habitats/results/hazard_ratio_table.xlsx


,Model,Cohort,Threshold,Low_risk_n,High_risk_n,Low_risk_events,High_risk_events,Logrank_p,HR,HR_95CI_low,HR_95CI_high,Cox_p,Cox_status,HR_report
0,Clinical,internal_test,0.390000,76,20,14,10,4.545601e-03,3.050817,1.351612,6.886211,0.007245,ok,"3.05 (1.35, 6.89)$^{*}$"
1,Clinical,external_test,0.413615,25,39,2,15,7.856486e-03,5.797963,1.324499,25.380444,0.019644,ok,"5.80 (1.32, 25.38)$^{*}$"
2,C-radiomics,internal_test,0.283935,47,49,2,22,3.614517e-06,13.620432,3.197821,58.013310,0.000412,ok,"13.62 (3.20, 58.01)$^{*}$"
3,C-radiomics,external_test,0.292894,38,26,4,13,7.734207e-04,5.503824,1.789984,16.923099,0.002921,ok,"5.50 (1.79, 16.92)$^{*}$"
4,H-radiomics,internal_test,0.278320,63,33,7,17,4.380309e-06,6.090973,2.520203,14.721015,0.000060,ok,"6.09 (2.52, 14.72)$^{*}$"
5,H-radiomics,external_test,0.290715,37,27,3,14,1.450065e-04,7.715170,2.210396,26.929041,0.001357,ok,"7.72 (2.21, 26.93)$^{*}$"
6,DL_3D,internal_test,0.425834,58,38,5,19,1.879463e-06,7.603831,2.833912,20.402270,0.000056,ok,"7.60 (2.83, 20.40)$^{*}$"
7,DL_3D,external_test,0.512995,42,22,5,12,1.015688e-04,6.103372,2.145538,17.362146,0.000696,ok,"6.10 (2.15, 17.36)$^{*}$"
8,fusion_soft_vote,internal_test,0.305003,50,46,1,23,3.843547e-08,33.581518,4.528925,249.003554,0.000587,ok,"33.58 (4.53, 249.00)$^{*}$"
9,fusion_soft_vote,external_test,0.330827,30,34,1,16,1.013042e-04,17.933194,2.374820,135.420526,0.005134,ok,"17.93 (2.37, 135.42)$^{*}$"
